In [2]:
import numpy as np
import pandas as pd
np.random.seed(42)

In [4]:

# Item catalog
items = pd.DataFrame({
    "item_id": range(20),
    "category": np.random.choice(
        ["main", "side", "dessert", "beverage"],
        20,
        p=[0.4, 0.25, 0.2, 0.15]
    ),
    "price": np.random.randint(80, 400, 20)
})

items.head()

,item_id,category,price
0,0,main,350
1,1,main,294
2,2,main,331
3,3,main,269
4,4,dessert,375


In [5]:
users = pd.DataFrame({
    "user_id": range(100),
    "avg_order_value": np.random.randint(200, 800, 100),
    "avg_items_per_order": np.random.randint(1, 5, 100),
    "price_sensitivity": np.random.rand(100)
})

users.head()

,user_id,avg_order_value,avg_items_per_order,price_sensitivity
0,0,671,1,0.485614
1,1,262,4,0.448424
2,2,338,2,0.994457
3,3,698,2,0.175925
4,4,792,2,0.018075


In [6]:
orders = []

for order_id in range(2000):
    user = np.random.choice(users.user_id)
    order_size = np.random.randint(1, 5)
    order_items = np.random.choice(items.item_id, order_size, replace=False)
    
    for position, item in enumerate(order_items):
        orders.append({
            "order_id": order_id,
            "user_id": user,
            "position": position,
            "item_id": item
        })

orders = pd.DataFrame(orders)
orders.head()

,order_id,user_id,position,item_id
0,0,57,0,14
1,0,57,1,12
2,0,57,2,3
3,1,25,0,5
4,1,25,1,7


In [7]:
training_rows = []

for order_id, group in orders.groupby("order_id"):
    group = group.sort_values("position")
    
    cart_items = []
    
    for _, row in group.iterrows():
        if len(cart_items) > 0:
            training_rows.append({
                "user_id": row.user_id,
                "cart_items": cart_items.copy(),
                "target_item": row.item_id
            })
        
        cart_items.append(row.item_id)

training_df = pd.DataFrame(training_rows)
training_df.head()

,user_id,cart_items,target_item
0,57,[14],12
1,57,"[14, 12]",3
2,25,[5],7
3,25,"[5, 7]",1
4,25,"[5, 7, 1]",2


In [8]:
from collections import defaultdict

co_matrix = defaultdict(lambda: defaultdict(int))

for order_id, group in orders.groupby("order_id"):
    item_list = list(group.sort_values("position").item_id)
    
    for i in range(len(item_list)):
        for j in range(i+1, len(item_list)):
            co_matrix[item_list[i]][item_list[j]] += 1

co_prob = {}

for i in co_matrix:
    total = sum(co_matrix[i].values())
    co_prob[i] = {
        j: co_matrix[i][j] / total
        for j in co_matrix[i]
    }

In [9]:
ranking_data = []

for _, row in training_df.iterrows():
    user = row.user_id
    cart = row.cart_items
    positive = row.target_item
    
    # Candidate set = co-occurrence from last item
    last_item = cart[-1]
    candidates = list(co_prob.get(last_item, {}).keys())
    
    # Add some random negatives
    random_negatives = list(
        np.random.choice(items.item_id, 5)
    )
    
    candidates = list(set(candidates + random_negatives))
    
    for candidate in candidates:
        ranking_data.append({
            "user_id": user,
            "cart_size": len(cart),
            "cart_value": items[items.item_id.isin(cart)].price.sum(),
            "candidate_item": candidate,
            "candidate_price": items.loc[
                items.item_id == candidate, "price"
            ].values[0],
            "label": 1 if candidate == positive else 0
        })

ranking_df = pd.DataFrame(ranking_data)
ranking_df.head()

,user_id,cart_size,cart_value,candidate_item,candidate_price,label
0,57,1,236,0,350,0
1,57,1,236,1,294,0
2,57,1,236,2,331,0
3,57,1,236,3,269,0
4,57,1,236,4,375,0


In [10]:
ranking_df["price_diff"] = (
    ranking_df["candidate_price"] -
    ranking_df["cart_value"] / ranking_df["cart_size"]
)

In [12]:
from sklearn.model_selection import train_test_split

X = ranking_df.drop("label", axis=1)
y = ranking_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[test_data],
    num_boost_round=100
)

[LightGBM] [Info] Number of positive: 2363, number of negative: 43315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000379 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 644
[LightGBM] [Info] Number of data points in the train set: 45678, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.051732 -> initscore=-2.908567
[LightGBM] [Info] Start training from score -2.908567


In [14]:
from sklearn.metrics import roc_auc_score

preds = model.predict(X_test)
auc = roc_auc_score(y_test, preds)

print("AUC:", auc)

AUC: 0.4898498375331055


In [15]:
ranking_df["pred"] = model.predict(X)

precision_at_5 = ranking_df.sort_values("pred", ascending=False) \
                            .head(5)["label"].mean()

print("Precision@5:", precision_at_5)

Precision@5: 0.8


In [16]:
def recommend(user_id, cart_items, top_n=5):
    last_item = cart_items[-1]
    candidates = list(co_prob.get(last_item, {}).keys())
    
    data = []
    
    for candidate in candidates:
        data.append({
            "user_id": user_id,
            "cart_size": len(cart_items),
            "cart_value": items[items.item_id.isin(cart_items)].price.sum(),
            "candidate_item": candidate,
            "candidate_price": items.loc[
                items.item_id == candidate, "price"
            ].values[0],
        })
    
    df = pd.DataFrame(data)
    df["price_diff"] = (
        df["candidate_price"] -
        df["cart_value"] / df["cart_size"]
    )
    
    df["score"] = model.predict(df)
    
    return df.sort_values("score", ascending=False) \
             .head(top_n)

In [17]:
recommend(user_id=1, cart_items=[3])

,user_id,cart_size,cart_value,candidate_item,candidate_price,price_diff,score
2,1,1,269,6,287,18.0,0.054405
16,1,1,269,5,292,23.0,0.053310
14,1,1,269,11,331,62.0,0.052426
12,1,1,269,8,132,-137.0,0.052393
15,1,1,269,13,120,-149.0,0.052393
